In [1]:
from dataclasses import dataclass
import os
from pathlib import Path
import tomllib
import pandas as pd
import geopandas as gpd
from sqlalchemy import create_engine
from nvi_etl.geo_reference import (
    pull_city_boundary, 
    pull_council_districts, 
    pull_zones, 
    pull_cdo_boundaries,
)
from openpyxl import Workbook
from openpyxl.utils.dataframe import dataframe_to_rows


WORKING_DIR = Path(os.getcwd()).parent.parent


with open(WORKING_DIR / "config.toml", "rb") as f:
    config = tomllib.load(f)


def get_engine(db_name=None):
    user = config["db"]["user"]
    password=config["db"]["password"]
    database=db_name or config["db"]["database"]
    host=config["db"]["host"]

    return create_engine(f"postgresql+psycopg://{user}:{password}@{host}:5432/{database}")

In [2]:
frame = pd.read_csv("Q:\\3_Projects\\NVI\\2025\\nvi_survey_data_2025_20260226.csv")
geoframe = gpd.read_file("Q:\\3_Projects\\NVI\\2025\\Final Shapefiles\\Final2025NVIDataset_cleaned_20260304.shp")
datadictionary = pd.read_excel(".\\conf\\nvi_answer_key_20260225.xlsx")

C:\Users\mike\AppData\Local\Temp\ipykernel_6116\2983482058.py:1: DtypeWarning: Columns (21) have mixed types. Specify dtype option on import or set low_memory=False.
  frame = pd.read_csv("Q:\\3_Projects\\NVI\\2025\\nvi_survey_data_2025_20260226.csv")


In [ ]:
geocoded = gpd.GeoDataFrame(
    frame.rename(
        columns={
            "Response ID": "response_id",
        }
    ).merge(
        geoframe.rename(
            columns={
                "Response_I": "response_id",
            }
        )[["response_id", "geometry"]], 
        on="response_id"
    )
)

In [4]:
DISTRICTS_YEAR = 2026
ZONES_YEAR = 2026

city = pull_city_boundary()
districts = pull_council_districts(DISTRICTS_YEAR)
zones = pull_zones(ZONES_YEAR)
cdos = pull_cdo_boundaries()

In [5]:

# # NOTE SC: Commented out joins for zones and districts for now 
# # NOTE MV: I've uncommented, because we can allow this to flow into the full response, 
# # and then filter later. We have plans to include the CDOs in the database.
geocoded = (
    geocoded
    .to_crs(2898)
    .sjoin(districts[["geometry", "district_number"]], how="left", predicate="within")
    .drop(columns="index_right")
)

geocoded = (
    geocoded
    .to_crs(2898)
    .sjoin(zones[["geometry", "zone_id"]], how="left", predicate="within")
)

In [6]:
# Let's make sense sure we didn't lose any rows
assert len(frame) == len(geocoded)

In [7]:
@dataclass
class Table:
    topic: str | None
    question: str 
    table: pd.DataFrame | None = None
    summary_level: str | None = None


def make_table(column, table=None, summary_level=None):
    return Table(
        column["topic_text"] or None,
        column["question_text"],
        table,
        summary_level=summary_level,
    )

In [ ]:
result = {}
errors = []
# Loop through each question
for i, column in datadictionary.drop_duplicates(subset="full_column").iterrows():
    tables = []

    topic_text = column["topic_text"]
    column_name = column["full_column"].replace("'", "’")

    if column_name not in frame:
        errors.append(column_name)
        continue

    labels = datadictionary[
        datadictionary["full_column"] == column_name
    ][["survey_code", "answer"]].set_index("survey_code")

    # First aggregate city-wide
    out = (
        geocoded[column_name]
        .value_counts(dropna=False)
        .to_frame()
    )

    try:
        out.index = out.index.astype(pd.Int64Dtype())
    except ValueError:
        errors.append("citywide")
        continue

    out = (
        out
        .join(labels)
        .sort_values(column_name)
        .reset_index(drop=True)
    )[["answer", "count"]]

    tables.append(make_table(column, out, "citywide"))


    # Then aggregate districts, zones

    for summary in ("district_number", "zone_id"):
        out = (
            geocoded[[column_name, summary]]
            .value_counts(dropna=False)
            .reset_index(summary)
        )

        out.index = out.index.astype(pd.Int64Dtype())

        out = (
            out
            .join(labels)
            .sort_values([summary, column_name])
            .reset_index(drop=True)
        )[[summary, "answer", "count"]]

        tables.append(make_table(column, out, summary))

    result[column["question_text"]] = tables

In [157]:
wb = Workbook()
toc = wb.active
toc.title = "Table of Contents"
toc.append(["Sheet #", "Question"])

for sheet_num, (question, tables) in enumerate(result.items(), 1):
    ws = wb.create_sheet(title=f"Q{sheet_num}")
    toc.append([sheet_num, question])
    first = tables[0]
    if first.topic:
        ws.append([first.topic])
    ws.append([first.question])
    for table in tables:
        if table.summary_level:
            ws.append([table.summary_level])
        for r in dataframe_to_rows(table.table, index=False, header=True):
            ws.append(r)
        ws.append([])

wb.save('survey_output.xlsx')

In [159]:
len(errors)

87

In [14]:
SEARCH_TERM = "that I won"

[   
    c for c in geocoded.columns
    if SEARCH_TERM.lower() in c.lower()
]

['Concern that I won’t be taken seriously, or no action will be taken:Hesitate_Reporting_Crime_Reason']

In [9]:
errors

['Other (please describe):Community_Engagement_Process_Participation',
 'Cleaned up or improved lot(s) that I do not own:In_The_Last_12_Months',
 'Cleaned up or improved lot(s) that I own:In_The_Last_12_Months',
 'Employment_Impacted_By_COVID',
 "Yes, I can access the internet at home but I'm unsure which of these apply:Internet_At_Home",
 "Concern that I won't be taken seriously, or no action will be taken:Hesitate_Reporting_Crime_Reason",
 'Other:Hesitate_Reporting_Crime_Reason',
 "Devices that don't work well:Issues_Prevent_Use_Device",
 'SchoolProgram_Leadership_Participation:Youth_In_Household_Last_12_Months_Questions',
 'SchoolProgram_Sports_Tutoring_Participation:Youth_In_Household_Last_12_Months_Questions',
 'Childcare_Prevent_Work_Meetings_Appointments',
 'citywide',
 'Friends, neighbors, etc.:Churches:DigitalEquity_Sources_Information',
 'Online search:Churches:DigitalEquity_Sources_Information',
 'Social media:Churches:DigitalEquity_Sources_Information',
 'Newspaper:Churches